Elizabeth Van Der Schaaf

US Retail Sales Dashboard

2/16/2025

### Background

The data for these dashboards was taken from the Monthly Trade Report complied by the US Census (US Census Bureau, 2025). It contains retail sales data going back to 1992 broken down by many different categories. I chose this data because I thought it would yield some interesting insights into American spending, especially around the role of e-commerce and the COVID pandemic.

I want to tell the story of what Americans are spending their money on and how spending patterns have changed over the years. I can use the detailed sales data broken down by month, year, and category to draw out trends or patterns in spending. My audience would be anyone in the retail sector, but especially those interested in trends associated with e-commerce. These may be people considering a shift from physical retail or expanding their operations.

This audience is interested in any major shifts in e-commerce and physical retail. They also would want to know which retail categories Americans tend to spend more on and any current trends. They may already have a general idea of some trends but may not have taken a closer look at the numbers. They may also not be aware of sales figures for certain retail categories and how these compare to overall retail spending.

The big idea of my dashboard is that e-commerce took off durning the pandemic and shows no signs of slowing down while some other sectors, especially department stores have shown no growth or a steady decline.

I designed two dashboards. The first (*How are Americans spending their money?*) contains a slider to allow the user to filter the data by year. There is a grouped bar chart and a line chart which show the performance of certain categories relative to one another. Annotations are included for a couple of years (2020 and 2024) to point out key findings. There is a link included at the bottom of this dashboard to the data source.

The second dashboard (*10-Year Trends*) is designed to show the user longer-term trends in the data. There is a line graph and a grouped bar chart which span the period of 2015 - 2024. The user can filter both graphs based on selected categories independently of one another. This was done to enhance the interactivity of the dashboards and ease exploration of the data. Annotations were included for a couple of key categories (*e-shopping* and *department*). It is possible to naviagte between the dashboards by using a link at the top.

### Reference
US Census Bureau. (2025, February 14). *Monthly Retail Trade* - Sales report. https://www.census.gov/retail/sales.html

In [ ]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objects as go
import pandas as pd

# read in data
data = pd.read_csv('us_retail.csv')

all_categories = data.category.unique()
selected_categories=['e-shopping','department','grocery','clothing&access','furniture','hobbies','electronics','superstores']
monthly_categories=['e-shopping','department','grocery','clothing&access']

# Filter data based on selected categories
filtered_data = data[data['category'].isin(selected_categories)]
filtered_monthly = data[data['category'].isin(monthly_categories)]

# Group by category and year, and calculate total sales for each group
sales_per_category_year = filtered_data.groupby(['category', 'year'])['sales'].sum().reset_index()
# Do same for total retail sales
retail_total_data = data[data['category'] == 'retail_total']
retail_total_per_year = retail_total_data.groupby(['year'])['sales'].sum().reset_index()
# Get list of years present in the data
years = sorted(sales_per_category_year['year'].unique().tolist())

# Choose colors for categories
color_map = {
    'e-shopping': 'chocolate',
    'department': 'cornflowerblue',
    'grocery': 'lightsteelblue',
    'clothing&access': 'mediumpurple',
    'furniture': 'powderblue',
    'hobbies': 'lavender',
    'electronics': 'lightblue',
    'superstores': 'lightskyblue',
}

# -----------------
# Create the Dash app
app = dash.Dash(__name__, suppress_callback_exceptions=True)

#-------------
# Bar chart filtered by year slider
filtered_bars = go.Figure()
@app.callback(
    Output('filtered-bars', 'figure'),
    [Input('year-slider', 'value')]
)
def update_filtered_bars(selected_year):
    filtered_bars = go.Figure() 
    selected_year = str(selected_year)
    
    # Filter data for the selected year and categories
    sales = sales_per_category_year[sales_per_category_year['year'] == int(selected_year)]
    
    # Add traces for selected categories
    for cat in selected_categories:
        # Filter sales data for the current category and the selected year
        category_sales = sales[sales['category'] == cat]
        
        # Merge with total retail sales to calculate the percentage
        merged_data = category_sales.merge(retail_total_per_year, on='year', suffixes=('', '_total'))
        merged_data['percentage_of_total'] = (merged_data['sales'] / merged_data['sales_total'])  # Convert to percentage
        
        # Add trace for the current category
        filtered_bars.add_trace(go.Bar(
            x=[selected_year],  # Display only for selected year
            y=merged_data['percentage_of_total'],
            name=cat,
            marker=dict(color=color_map.get(cat, "gray")),
            text=merged_data['category'],  # Add category text as label
            textposition='outside',  # Position the text outside the bars
            hoverinfo='x+y+text' 
        ))

    # Update layout and add title and axis labels
    filtered_bars.update_layout(
        title="Sales as a percentage of total retail sales",
        title_font=dict(family='Arial, sans-serif', size=24),
        font=dict(family='Arial, sans-serif', size=14),
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(
            title_font=dict(family='Arial, sans-serif', size=16),
            tickfont=dict(family='Arial, sans-serif', size=16),
            showline=True,
            linecolor='grey'
        ),
        yaxis=dict(
            title="Percent of Total Retail Sales",
            tickformat=".0%",
            title_font=dict(family='Arial, sans-serif', size=16),
            tickfont=dict(family='Arial, sans-serif', size=12),
            showline=True,
            linecolor='grey'
        ),
        showlegend=False
    )

    # Remove gridlines
    filtered_bars.update_xaxes(showgrid=False)
    filtered_bars.update_yaxes(showgrid=False)
    
    return filtered_bars

#------------------
# Line chart filtered by year slider
detailed_lines = go.Figure()
@app.callback(
    Output('monthly-variation-graph', 'figure'),
    [Input('year-slider', 'value')]
)
def update_monthly_variation(selected_year):
    detailed_lines = go.Figure()
    selected_year = int(selected_year)
    
    # Filter data for the selected year and categories
    filtered_monthly_year = filtered_monthly[filtered_monthly['year'] == selected_year]
    
    # Group by category, year, and month to calculate total sales by month
    monthly_sales_per_category = filtered_monthly_year.groupby(['category', 'month'])['sales'].sum().reset_index()
    
    # Add a line for each selected category
    for cat in monthly_categories:
        # Filter data for the current category
        category_sales = monthly_sales_per_category[monthly_sales_per_category['category'] == cat]
        
        # Add line trace for the current category
        detailed_lines.add_trace(go.Scatter(
            x=category_sales['month'],  
            y=category_sales['sales'],  
            mode='lines+text', 
            name=cat,  # Legend label
            marker=dict(color=color_map.get(cat, "gray")),
            line=dict(width=2),  
            text=[cat if i == len(category_sales) - 1 else '' for i in range(len(category_sales))],
            textposition='top left', 
        ))
    # Add annotations for e-shopping and dept stores in April 2020
    if selected_year == 2020:  # Only show annotation for 2020
        e_shopping_data = monthly_sales_per_category[(monthly_sales_per_category['category'] == 'e-shopping')
        & (monthly_sales_per_category['month'] == 4)]
        department_data = monthly_sales_per_category[(monthly_sales_per_category['category'] == 'department')
        & (monthly_sales_per_category['month'] == 4)]
        
        if not e_shopping_data.empty:
            april_esales = e_shopping_data['sales'].values[0]
            april_dept = department_data['sales'].values[0]
            
            detailed_lines.update_layout(
                annotations=[
                    dict(
                        x=4,  # X position - April
                        y=april_esales,  # Y position - sales for e-shopping in April
                        text="COVID pandemic pushes e-shopping over groceries",
                        showarrow=True,
                        arrowhead=2,  # Arrow style
                        ax=0,  # X offset of the arrow
                        ay=-40,  # Y offset of the arrow
                        font=dict(size=15, color='black'),
                        bgcolor='papayawhip',
                        borderpad=4
                    ),
                    dict(
                        x=4,  # X position - April
                        y=april_dept,  # Y position - sales for e-shopping in April
                        text="Other retail categories suffer",
                        showarrow=True, 
                        arrowhead=2,  # Arrow style
                        ax=0,  # X offset of the arrow
                        ay=-40,  # Y offset of the arrow
                        font=dict(size=15, color='black'),
                        bgcolor='papayawhip',
                        borderpad=4
                    )
                ]
            )
    # Annotation for e-shopping in 2024
    if selected_year == 2024: 
        e_shopping_data = monthly_sales_per_category[(monthly_sales_per_category['category'] == 'e-shopping')
        & (monthly_sales_per_category['month'] == 7)]
        
        if not e_shopping_data.empty:
            july_sales = e_shopping_data['sales'].values[0]
            
            detailed_lines.update_layout(
                annotations=[
                    dict(
                        x=7, 
                        y=july_sales,
                        text="Post-pandemic: e-shopping remains strong", 
                        showarrow=True, 
                        arrowhead=2,  # Arrow style
                        ax=0,  # X offset of the arrow
                        ay=-40,  # Y offset of the arrow
                        font=dict(size=15, color='black'),
                        bgcolor='papayawhip',
                        borderpad=4
                    )
                ]
            )
    # Update layout and add title and axis labels
    detailed_lines.update_layout(
        title=f"Sales in select categories for {selected_year}",
        title_font=dict(family='Arial, sans-serif', size=24),
        font=dict(family='Arial, sans-serif', size=14),
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(
            #title="Month",
            title_font=dict(family='Arial, sans-serif', size=16),
            tickfont=dict(family='Arial, sans-serif', size=12),
            tickmode='array',
            tickvals=list(range(1, 13)), 
            ticktext=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
            showline=True,
            linecolor='grey'
        ),
        yaxis=dict(
            title="Sales",
            tickformat="$,",
            title_font=dict(family='Arial, sans-serif', size=16),
            tickfont=dict(family='Arial, sans-serif', size=12),
            showline=True,
            linecolor='grey'
        ),
        showlegend=False
    )

    # Remove gridlines
    detailed_lines.update_xaxes(showgrid=False)
    detailed_lines.update_yaxes(showgrid=False)

    return detailed_lines

#------------------
# Define layout for first dash
dash_1_layout = html.Div([
    html.H1("US Retail Sales | How are Americans spending their money?",
           style={'fontFamily':'Arial','fontSize':'30px','fontWeight':'bold',
                  'color':'floralwhite','backgroundColor':'dimgrey',
                  'marginTop':'2px','marginBottom': '5px'}),
    html.P("Use the slider to filter by year.",
           style={'fontFamily':'Arial','fontSize':'16px',
                  'color':'floralwhite','backgroundColor':'dimgrey','padding':'2px',
                  'marginTop': '5px'}),
    
    # Slider for year selection
    dcc.Slider(
        id='year-slider',
        min=min(years),
        max=max(years),
        step=1,  # 1 year at a time
        marks={year: str(year) for year in years},  # create marks based on years
        value=2020,  # default selection
    ),
    
    dcc.Graph(id='filtered-bars', figure=filtered_bars),
    dcc.Graph(id='monthly-variation-graph', figure=detailed_lines),
    
    # Div containing the external link
    html.Div(
        children=[
            html.A(
                "Link to data source - Monthly Retail Trade Report ",  
                href="https://www.census.gov/retail/sales.html", 
                target="_blank",  # Opens the link in a new tab
                style={
                    'fontFamily': 'Arial, sans-serif',
                    'fontSize': '16px',
                    'color': 'deepskyblue', 
                    #'textDecoration': 'underline',  # No underline by default
                    'marginTop': '10px',  # Add some spacing from the other content
                }
            )
        ],
        style={
            'display': 'flex',
            'justifyContent': 'left',  
            'padding': '10px',
            'backgroundColor': 'dimgrey',  # Background color of the div
            'borderRadius': '10px',  # Optional rounded corners for the bottom
            'marginTop': '10px',  # Space between the link and other content
        }
    )
],
    style={
        'display': 'flex',              
        'flexDirection': 'column',        # Stack vertically
        'justifyContent': 'center',       
        'fontFamily': 'Arial',  # Fixed typo from 'fontFamiliy' to 'fontFamily'
        'fontSize': '12px',
        'color': 'floralwhite',
        'backgroundColor': 'dimgrey',  # Background color for entire div
        'padding': '8px',                # Padding inside the div
        'borderRadius': '10px'           # Optional: rounded corners
    })



# ------------------------
# Line chart filtered by category checklist
# Update function for the line chart
line_chart = go.Figure()
@app.callback(
    Output("line_chart", "figure"),
    [Input("checklist", "value")]
)
def update_line_chart(categories):
    # Filter data based on selected categories
    mask = data.category.isin(categories)
    filtered_data = data[mask]
    # Create figure
    line_chart = go.Figure()
    # Plot each category as a line
    for category in categories:
        category_data = filtered_data[filtered_data['category'] == category]
        line_chart.add_trace(go.Scatter(
            x=category_data['year'],
            y=category_data['sales'],
            line=dict(color=color_map.get(category,"gray")),
            name=f"{category}",
            mode='lines'
        ))

        # Add annotation for department stores
    if 'department' in categories:
        department_data_2018 = filtered_data[(filtered_data['category'] == 'department') 
        & (filtered_data['year'] == 2018)]
        if not department_data_2018.empty:
            department_sales_2018 = department_data_2018['sales'].values[0]  
            line_chart.add_annotation(
                x=2018,  
                y=department_sales_2018, 
                text="Department store sales have faced a steady decline", 
                showarrow=True,  
                arrowhead=2,  
                ax=0,  
                ay=-40, 
                font=dict(family='Arial, sans-serif', size=14, color='black'),
                bgcolor='papayawhip',  
                borderpad=4  
            )
    # Update layout and add title and axis labels
    line_chart.update_layout(
        title="E-commerce took off during the pandemic <br>Shows no sign of stopping",
        title_font=dict(family='Arial, sans-serif', size=24), 
        font=dict(family='Arial, sans-serif', size=14),
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(
            #title="Year",
            title_font=dict(family='Arial, sans-serif', size=16),
            tickfont=dict(family='Arial, sans-serif', size=12),
            showline=True,
            linecolor='grey'
        ),
        yaxis=dict(
            title="Sales",
            tickformat="$,",
            title_font=dict(family='Arial, sans-serif', size=16),
            tickfont=dict(family='Arial, sans-serif', size=12),
            showline=True,
            linecolor='grey'
         ),
            

    )
    # Remove gridlines
    line_chart.update_xaxes(showgrid=False)
    line_chart.update_yaxes(showgrid=False)
    return line_chart

# ---------------
# Bar chart filtered by category dropbox
grouped_bar = go.Figure()
@app.callback(
    Output('grouped-bar', 'figure'),
    [Input('dropdown', 'value')]
)
def update_grp_bar(selected_cats):
    grouped_bar = go.Figure()

    # Add traces for selected categories
    # Calculated as a percentage of total retail sales
    for cat in selected_cats:
        # Filter sales data for current category
        category_sales = sales_per_category_year[sales_per_category_year['category'] == cat]
        
        # Merge with total retail sales to calculate the percentage
        merged_data = category_sales.merge(retail_total_per_year, on='year', suffixes=('', '_total'))
        merged_data['percentage_of_total'] = (merged_data['sales'] / merged_data['sales_total'])
        
        # Add trace for current category
        grouped_bar.add_trace(go.Bar(
            x=merged_data['year'],
            y=merged_data['percentage_of_total'],
            name=cat,
            marker=dict(color=color_map.get(cat,"gray")),
        ))
                # Add annotation: e-shopping in 2024
        if cat == 'e-shopping':
            es_2024_sales = merged_data[merged_data['year'] == 2024]['percentage_of_total'].values
            if len(es_2024_sales) > 0:
                grouped_bar.add_annotation(
                    x=2024,
                    y=es_2024_sales[0],  # percentage for e-shopping in 2024
                    text="In 2024 e-shopping reached <br><b>18%</b> of total retail sales", 
                    showarrow=True,
                    arrowhead=2,
                    ax=0,
                    ay=-40,
                    font=dict(size=15, color='black'),
                    bgcolor='papayawhip',
                    borderpad=4
                )

    min_year = merged_data['year'].min()
    max_year = merged_data['year'].max()

    # Update layout and add title and axis labels
    grouped_bar.update_layout(
        title="Sales as a percentage of total retail sales",
        title_font=dict(family='Arial, sans-serif', size=24), 
        font=dict(family='Arial, sans-serif', size=14),
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(
            #title="Year",
            title_font=dict(family='Arial, sans-serif', size=16),
            tickfont=dict(family='Arial, sans-serif', size=12),
            showline=True,
            linecolor='grey',
        ),
        yaxis=dict(
            title="Percent of Total Retail Sales",
            tickformat=".0%",
            title_font=dict(family='Arial, sans-serif', size=16),
            tickfont=dict(family='Arial, sans-serif', size=12),
            showline=True,
            linecolor='grey'
         ),
            

    )
    # Remove gridlines
    grouped_bar.update_xaxes(showgrid=False)
    grouped_bar.update_yaxes(showgrid=False)
    return grouped_bar

# ------------------------
dash_2_layout = html.Div([
    html.H1("US Retail Sales | 10-Year Trends",
           style={'fontFamily':'Arial','fontSize':'30px','fontWeight':'bold',
                 'color':'dimgrey',
                  'margintop':'2px','marginBottom': '5px'}),
    html.P("Use filters to examine retail sales by category.",
          style={'fontFamily':'Arial','fontSize':'16px',
                'color':'dimgrey','padding':'2px',
                 'marginTop': '5px'}),
        # Slider for year selection
    dcc.Checklist(
        id='checklist',
        options=[{'label': x, 'value': x} for x in selected_categories],
        value=selected_categories[:4],  # Default selected categories
        labelStyle={'display': 'inline-block'},
        style={'font-family': 'Arial, sans-serif'}
    ),
    dcc.Graph(id='line_chart', figure=line_chart),
    dcc.Dropdown(
        id='dropdown',
        options=[{'label': cat, 'value': cat} for cat in selected_categories],
        value=selected_categories[:3], 
        multi=True,
        style={'font-family': 'Arial, sans-serif'}
    ),
    dcc.Graph(id='grouped-bar', figure=grouped_bar)
],)

#-------------------
# Define main layout with navigation links and page content
app.layout = html.Div([
    dcc.Location(id='url', refresh=False),  # Location component to track the URL
    html.Div(id='page-links'),  # This will dynamically update the displayed links
    html.Div(id='page-content'),  # This will update based on URL path
])
# Callback to update the displayed links based on the current page
@app.callback(
    Output('page-links', 'children'),
    [Input('url', 'pathname')]
)
def update_links(pathname):
    if pathname == '/page-1':
        return html.Div([
            dcc.Link('Go to 10-Year Trends Dashboard', href='/page-2',
                     style={'fontFamily': 'Arial', 'fontSize': '18px'})
        ])
    elif pathname == '/page-2':
        return html.Div([
            dcc.Link('Go to American Spending Dashboard', href='/page-1',
                     style={'fontFamily': 'Arial', 'fontSize': '18px'})
        ])
    else:
        return html.Div([
            dcc.Link('Go to 10-Year Trends Dashboard', href='/page-2',
                     style={'fontFamily': 'Arial', 'fontSize': '18px'})
        ])

# Callback to switch between pages based on URL
@app.callback(
    Output('page-content', 'children'),
    [Input('url', 'pathname')]
)
def display_page(pathname):
    if pathname == '/page-1':
        return dash_1_layout 
    elif pathname == '/page-2':
        return dash_2_layout 
    else:
        return dash_1_layout  # default to dash 1
        
if __name__ == '__main__':
    app.run(debug=True, jupyter_mode="tab", use_reloader=False)